# Parameter-Controlled Archival Nuclear Test Video Generator
### Two-Phase LoRA Training Pipeline (Kaggle T4 / P100 GPU)

**Pipeline Architecture:**
1. **Phase 1 (Spatial LoRA):** Fine-tunes SD 1.5 UNet spatial self/cross-attention projections (`to_k`, `to_v`, `to_q`, `to_out.0`) on static frames to ground visual appearance:
   - Burst geometry and cloud morphology
   - Nevada Test Site desert vs. Pacific Proving Grounds ocean
   - Historical 16mm film grain, exposure, and color degradation
2. **Phase 2 (Motion LoRA):** Freezes spatial UNet (with Spatial LoRA active), fine-tunes AnimateDiff temporal *self-attention* layers on 16-frame latent sequences (`4, 16, 64, 64`) to learn fluid explosion dynamics:
   - Shockwave propagation & Wilson cloud flash
   - Toroidal vortex expansion & stem suction
   - Base surge progression in underwater/shallow bursts
3. **Stage 6 Evaluation Suite:** Multi-task 3D classifier measuring control accuracy against majority-class baselines.



In [ ]:
# Step 1: Environment Setup & Package Installation
!pip install -q diffusers transformers accelerate peft pyyaml scenedetect opencv-python imageio imageio-ffmpeg tqdm
!nvidia-smi



In [ ]:
# Step 2: GPU and PyTorch Verification
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")



In [ ]:
# Step 3: Project Setup in Kaggle Working Directory
import os
import shutil

WORKDIR = "/kaggle/working/Nuclear-Test-Video-Generator"
if not os.path.exists(WORKDIR):
    # Clone or link project code
    !git clone https://github.com/v1has/Nuclear-Test-Video-Generator.git {WORKDIR} || mkdir -p {WORKDIR}

os.chdir(WORKDIR)
print("Current Working Directory:", os.getcwd())
!ls -lh



In [ ]:
# Step 4: Pre-encode Latents on GPU (if not already cached)
# GPU pre-encoding takes ~100-200 ms per clip, saving gigabytes of per-epoch compute
!python scripts/04_preencode_latents.py --all --device cuda



In [ ]:
# Step 5: Phase 1 (Spatial LoRA) Training
# Trains spatial cross-attention layers on static frames
!python training/01_train_spatial_lora.py \
    --config training/config.yaml \
    --max_train_steps 2000 \
    --device cuda



In [ ]:
# Step 6: Phase 2 (Motion LoRA) Training
# Freezes spatial UNet + Spatial LoRA, trains temporal self-attention in AnimateDiff motion module
!python training/02_train_motion_lora.py \
    --config training/config.yaml \
    --max_train_steps 3000 \
    --device cuda



In [ ]:
# Step 7: Sample Video Generation Across Burst Types
!python inference/generate.py \
    --burst_type underwater \
    --yield_tier tactical-yield \
    --setting tropical-ocean \
    --phase expansion \
    --film_look colour-restored \
    --output samples/underwater_sample.mp4 \
    --device cuda

!python inference/generate.py \
    --burst_type surface-tower \
    --yield_tier intermediate-yield \
    --setting desert-flats \
    --phase expansion \
    --film_look high-contrast-monochrome \
    --output samples/surface_tower_sample.mp4 \
    --device cuda



In [ ]:
# Step 8: Stage 6 Quantitative Control Evaluation Suite
# Evaluates generated samples against trained 3D classifier to verify >25% margin over majority baseline
!python scripts/05_train_classifier.py --train --epochs 5 --device cuda
!python scripts/05_train_classifier.py --evaluate_synthetic --synthetic_dir samples/ --device cuda



In [ ]:
# Step 9: Package Checkpoints for Download
!zip -r /kaggle/working/nuclear_generator_lora_weights.zip \
    checkpoints/phase1_spatial/final_spatial_lora \
    checkpoints/phase2_motion/final_motion_lora \
    models/control_classifier.pt \
    data/synthetic_control_report.json

print("Package created at /kaggle/working/nuclear_generator_lora_weights.zip")

